# 02. Baseline: seberapa baik heuristik produksi?

**Pertanyaan:** pada gold set, seberapa baik skor indikasi AI di produksi memisahkan manusia dan AI, dan
berapa mahasiswa jujur yang **tertuduh** pada ambang yang dipakai sekarang?

Notebook ini menjalankan scorer asli dari `backend/apps/analysis/engines/` terhadap `gold_set.csv`.
Sinyal proses tidak ikut karena gold set hanya berisi teks, tanpa telemetri pengerjaan.

Empat jawaban yang dicari, berurutan:

1. ROC-AUC: seberapa jauh skor memisahkan dua kelas.
2. FPR pada ambang produksi (42 dan 70): berapa manusia yang dituduh.
3. Ambang yang menjaga FPR di bawah 5%.
4. Sinyal mana yang **mati** dan mana yang **terbalik**: sinyal yang tidak membedakan apa pun tetap menyumbang
   bobot ke setiap submission, sehingga hanya menaikkan skor semua orang secara merata.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import config

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)
config.ensure_dirs()
GOLD = config.GOLD_DIR / "gold_set.csv"

In [2]:
from src import evaluate_baseline as eb
from src import metrics, viz
from src.evaluation import CURRENT_HIGH_THRESHOLD, CURRENT_MID_THRESHOLD, MAX_ACCEPTABLE_FPR

eb.bootstrap_backend()
from apps.analysis.engines import ai_score

pd.DataFrame(
    {
        "bobot": ai_score.TEXT_WEIGHTS,
    }
).assign(ambang_mid=ai_score.MID_THRESHOLD, ambang_high=ai_score.HIGH_THRESHOLD).rename_axis("sinyal teks")

,bobot,ambang_mid,ambang_high
sinyal teks,,,
uniformity,0.40,42,70
formulaic_phrasing,0.25,42,70
impersonality,0.05,42,70
flat_certainty,0.05,42,70
lexical_uniformity,0.25,42,70


## Ilustrasi mekanisme (bukan evaluasi)

Dua paragraf pendek buatan tangan, hanya untuk memperlihatkan **bagaimana skor terbentuk**: setiap sinyal
bernilai 0 sampai 1, dikali bobot, dijumlahkan. Dua contoh ini tidak membuktikan apa pun tentang akurasi;
itulah gunanya gold set di bawah.

In [3]:
from apps.analysis.engines.text_features import extract_features

contoh = {
    "gaya mahasiswa (informal, personal)": (
        "Menurut saya masalahnya bukan di anggaran, tapi di cara daerah memakainya. Waktu saya magang di "
        "dinas, uangnya cair telat dan proyeknya jalan seadanya. Jadi ya, saya kurang yakin transfer saja "
        "cukup. Mungkin perlu pendampingan dulu."
    ),
    "gaya generik (seragam, formal)": (
        "Ketimpangan antarwilayah merupakan isu yang kompleks dan multidimensi. Selain itu, faktor "
        "infrastruktur memainkan peran yang sangat penting. Oleh karena itu, diperlukan pendekatan yang "
        "komprehensif dan berkelanjutan. Dengan demikian, kesejahteraan masyarakat dapat ditingkatkan secara "
        "optimal."
    ),
}
tabel = {}
for nama, teks in contoh.items():
    hasil = ai_score.score_ai_probability(extract_features(teks))
    tabel[f"{nama} → skor {hasil.score}"] = {s.key: s.value for s in hasil.breakdown}
pd.DataFrame(tabel).round(2)

,"gaya mahasiswa (informal, personal) → skor 35","gaya generik (seragam, formal) → skor 59"
uniformity,0.41,0.9
formulaic_phrasing,0.00,0.0
impersonality,0.33,1.0
flat_certainty,1.00,1.0
lexical_uniformity,0.50,0.5


## Tahap 1. Muat gold set dan skor semua teks

In [4]:
# Parameter. split=None memakai semuanya; "test" hanya split uji. mixed: "exclude" | "ai" | "human".
SPLIT = None
MIXED_AS = "exclude"

HAS_GOLD = GOLD.exists()
if not HAS_GOLD:
    print("gold_set.csv belum ada. Bangun dulu di notebook 01 (sisi manusia DAN sisi AI). Sel-sel di bawah dilewati.")
else:
    rows = eb.load_rows(GOLD, SPLIT, MIXED_AS)
    scores, labels, per_signal = eb.score_rows(rows)
    n_ai = sum(labels)
    print(f"{len(rows)} teks: {len(rows) - n_ai} manusia, {n_ai} AI")

gold_set.csv belum ada. Bangun dulu di notebook 01 (sisi manusia DAN sisi AI). Sel-sel di bawah dilewati.


## Tahap 2. Pemisahan: sebaran skor dan ROC-AUC

In [5]:
if HAS_GOLD and 0 < sum(labels) < len(labels):
    auc = metrics.roc_auc(scores, labels)
    print(f"ROC-AUC = {auc:.3f}", "(nyaris menebak)" if auc < 0.6 else "(arah terbalik!)" if auc < 0.5 else "")
    fig1 = viz.plot_score_distribution(scores, labels, mid=CURRENT_MID_THRESHOLD, high=CURRENT_HIGH_THRESHOLD)
    fig2 = viz.plot_roc({"heuristik produksi": (scores, labels)})
    plt.show()
elif HAS_GOLD:
    print("Hanya satu kelas pada gold set ini: metrik pemisahan tidak terdefinisi.")

## Tahap 3. Ambang: siapa yang tertuduh?

FPR adalah metrik paling penting secara etis: porsi tulisan manusia yang dituduh AI. Tabel berikut membaca
ambang produksi, lalu mencari ambang terendah yang masih menjaga FPR di bawah 5%. Itulah cara memilih
ambang yang bisa dipertanggungjawabkan; "70 karena angkanya bulat" tidak bisa dijawab saat ditanya berapa
mahasiswa jujur yang akan tertuduh.

In [6]:
if HAS_GOLD and 0 < sum(labels) < len(labels):
    baris = []
    for nama, ambang in (("mid", CURRENT_MID_THRESHOLD), ("high", CURRENT_HIGH_THRESHOLD)):
        r = metrics.confusion_at(scores, labels, ambang)
        baris.append({"ambang": f"{nama} (≥ {ambang})", "F1": r.f1, "presisi": r.precision, "recall": r.recall, "FPR": r.false_positive_rate})
    aman, r = metrics.threshold_at_max_fpr(scores, labels, max_fpr=MAX_ACCEPTABLE_FPR)
    baris.append({"ambang": f"aman, FPR < {MAX_ACCEPTABLE_FPR:.0%} (≥ {aman})", "F1": r.f1, "presisi": r.precision, "recall": r.recall, "FPR": r.false_positive_rate})
    display(pd.DataFrame(baris).set_index("ambang").round(3))
    print(f"Pada ambang aman {aman}: {r.false_positive} dari {r.false_positive + r.true_negative} tulisan manusia tetap tertuduh.")
    viz.plot_threshold_sweep(scores, labels, marks=(CURRENT_MID_THRESHOLD, CURRENT_HIGH_THRESHOLD))
    plt.show()

## Tahap 4. Diagnosis per sinyal

Korelasi positif berarti sinyal naik saat teks memang AI. **MATI** (|r| < 0,05): tidak membedakan apa pun,
layak dibuang. **TERBALIK** (r < -0,1): arahnya melawan label.

In [7]:
if HAS_GOLD and 0 < sum(labels) < len(labels):
    diag = eb.signal_diagnostics(per_signal, labels)
    display(pd.DataFrame(diag).set_index("sinyal").round(3))
    viz.plot_signal_correlations(diag)
    plt.show()

## Tahap 5. Pemeriksaan kebocoran panjang

Bila teks AI sistematis lebih panjang atau pendek, sebagian AUC berasal dari panjang, bukan gaya.

In [8]:
if HAS_GOLD and 0 < sum(labels) < len(labels):
    eb.print_length_check(rows, labels)

## Kesimpulan dan langkah berikutnya

* **Hasil tercatat** (17 Agustus 2026, 999 abstrak): sebelum tuning FPR pada ambang lama 0,838; sesudah tuning
  AUC set penuh 0,901 dan ambang `mid` 42 dengan FPR di bawah 5% (lihat notebook 03).
* Pada uji jalan awal, `impersonality` mati dan `mechanical_polish` justru sinyal terkuat: dugaan awal yang
  berasal dari dua esai buatan tangan dibantah data. Itu alasan tahap ini ada.
* **Batasan register.** Gold set berisi abstrak akademik, sedangkan ThinkPath menilai esai mahasiswa
  (penanda personal 0,210 per teks pada abstrak vs 1,859 pada esai mahasiswa). Sinyal yang diam di sini
  belum tentu diam pada esai. Yang boleh disimpulkan: mana yang lebih memisahkan manusia dari AI pada teks
  Indonesia register ini. Yang **tidak** boleh: bahwa itu akurasi ThinkPath pada jawaban mahasiswa.

Lanjut ke `03_tuning_bobot`.